# Credit Risk Analytics & Risk Modelling

**Educational portfolio project.** The included dataset is synthetic and the target labels are artificially generated. Metrics are not evidence of real-world credit-risk performance.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "credit_risk_synthetic.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "credit_risk_synthetic.csv"
VISUALS_DIR = ROOT / "visualizations"
VISUALS_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Data quality and overview

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))
display(df.describe(include="all").T)
print("Duplicate rows:", df.duplicated().sum())

## 2. Target distribution

In [ ]:
target_counts = df["default"].value_counts().sort_index()
ax = target_counts.plot(kind="bar", figsize=(7, 4), color="steelblue")
ax.set(title="Synthetic target distribution", xlabel="Demo default label (0/1)", ylabel="Record count")
ax.set_xticklabels([str(x) for x in target_counts.index], rotation=0)
plt.tight_layout()
plt.savefig(VISUALS_DIR / "target_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
print("Synthetic positive-label rate:", round(df["default"].mean(), 3))

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["annual_income"], bins=30, edgecolor="white")
axes[0].set_title("Annual income (synthetic)")
axes[0].set_xlabel("Illustrative income")
axes[1].hist(df["credit_score"], bins=25, edgecolor="white")
axes[1].set_title("Credit-score-like value")
axes[1].set_xlabel("Synthetic score")
axes[2].boxplot([df.loc[df["default"] == 0, "debt_to_income"], df.loc[df["default"] == 1, "debt_to_income"]], labels=["0", "1"])
axes[2].set_title("Debt-to-income by demo label")
axes[2].set_xlabel("Demo default label")
fig.tight_layout()
fig.savefig(VISUALS_DIR / "feature_distributions.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
numeric = df.select_dtypes(include="number").drop(columns=["applicant_id"], errors="ignore")
numeric = numeric.drop(columns=["applicant_id"], errors="ignore")
corr = numeric.corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.columns)), labels=corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8, label="Correlation")
ax.set_title("Numeric feature correlations (synthetic data)")
fig.tight_layout()
fig.savefig(VISUALS_DIR / "correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Baseline modelling

This section uses a stratified holdout split and preprocessing pipelines. The scores are for workflow demonstration only because the labels are synthetic.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["default", "applicant_id"])
y = df["default"]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")
}

In [ ]:
roc_fig, roc_ax = plt.subplots(figsize=(7, 5))
for name, estimator in models.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probabilities)
    print(f"\n{name}")
    print(classification_report(y_test, predictions, zero_division=0))
    print(f"ROC-AUC: {auc:.3f}")
    ConfusionMatrixDisplay.from_predictions(y_test, predictions, values_format="d")
    plt.title(f"{name} — synthetic demo data")
    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(VISUALS_DIR / f"confusion_matrix_{safe_name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    roc_ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

roc_ax.plot([0, 1], [0, 1], linestyle="--", label="Chance baseline")
roc_ax.set(title="ROC curves — synthetic demo data", xlabel="False positive rate", ylabel="True positive rate")
roc_ax.legend()
roc_fig.tight_layout()
roc_fig.savefig(VISUALS_DIR / "roc_curves.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved charts to:", VISUALS_DIR)

## 5. Conclusions and next steps

- Treat every pattern and metric here as illustrative because the data and target are synthetic.
- Replace the CSV only with a properly licensed and documented dataset.
- For a meaningful project, document provenance, validate data quality, test generalization, assess fairness, and explain model limitations.
- Do not use this template to make real lending decisions.